# Lecture 07 companion notebook — Python: pandas and seaborn

**BIOE 610 Computational Tools · Week 7**

Every example from Lecture 07 as cells you can run, change and break — packages, control flow, method chaining, `.melt()`, seaborn figures built up in layers, and the figure gallery (clustermap, growth curves, dose-response, PCA). There is a matching R notebook (`Lecture07_R_wrangling_figures.ipynb`) with the same sections in the same order — keep them open side by side to compare. Needs `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy` and (section 16 only) `scikit-learn`.

**How to use this notebook in Positron**

1. Open the file — Positron shows it as a notebook automatically
2. Pick a **Python interpreter** if prompted (top right)
3. Run a cell with **Shift-Enter**. Run them in order the first time; then go back and experiment — *change the numbers, break things on purpose, read the errors*.

## 1 · Installing vs. loading packages

`pip install` puts a package in the *active environment* — **once** per environment. `import` makes it available in *this* session — **every** session. `ModuleNotFoundError` means it is not installed *in this environment* (`which python3` tells you which one that is).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 2 · Inspecting objects

When an object surprises you, interrogate it first. These five answer most questions.

In [ ]:
stickle = pd.read_csv(
    "https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv",
    sep="\t")

type(stickle)
print(stickle.shape)
list(stickle.columns[:6])

In [ ]:
stickle.iloc[:, :5].info()        # columns, dtypes, non-null counts
stickle["Gene1"].describe()       # quick summary of one column

## 3 · Logical operators and membership

Comparisons return boolean arrays; boolean arrays select rows. New today: **membership** with `np.isin()` / `.isin()`. In pandas/numpy use `&`, `|`, `~` with **parentheses** around each comparison.

In [ ]:
x = np.array([1, 5, 10, 15, 20])
x > 10                      # one True/False per element
x[x > 10]                   # keep only where True - L05's boolean indexing
np.isin(x, [1, 15, 99])     # membership: is each element in the set?

## 4 · Control flow: making decisions with `if`

Python's blocks are the **indentation** (4 spaces), and `else if` becomes `elif`. The vectorized cousin `np.where()` gives one answer per element — no loop needed.

In [ ]:
mass = 1.42
if mass > 1.0:
    print("big fish")          # the indented lines ARE the block
elif mass > 0.5:
    print("medium fish")
else:
    print("small fish")

In [ ]:
x = np.array([0.3, 0.8, 1.42])
np.where(x > 1.0, "big", "small")    # vectorized: one answer per element

## 5 · Control flow: repeating with `for` (and `while`)

Both languages loop directly over a collection's *elements* — no counters required. `enumerate()` gives position **and** element (0-based!). `break` leaves early; `continue` skips to the next turn.

In [ ]:
genes = ["tp53", "brca1", "myc"]
for g in genes:
    print(g.upper())

for i, g in enumerate(genes):     # position AND element (0-based!)
    print(i, g)

In [ ]:
i = 1
while i < 8:                      # repeat until the condition fails
    i = i * 2
print(i)

## 6 · When you do *not* need a loop

Vectorized code is shorter, faster and easier to read. Loops earn their keep for files, simulations, and step-*n*-depends-on-step-*n−1* problems.

In [ ]:
x = np.array([2, 3, 4, 5])
print(x**2)                                   # not: for i in range(len(x)) ...
genes_cols = [c for c in stickle.columns if c.startswith("Gene")]
stickle[genes_cols].sum(axis=1).head()        # not: nested loops

## 7 · Method chaining: pandas' pipe

Each method takes a DataFrame and returns a DataFrame, so they chain with the dot. Wrap the chain in `( )` to put each step on its own line.

In [ ]:
mydata = pd.DataFrame({
    "hydrogel_concentration": pd.Categorical(["low", "high", "high", "high",
                                              "medium", "medium", "medium", "low"]),
    "compression":  [3.4, 3.4, 8.4, 3, 5.6, 8.1, 8.3, 4.5],
    "conductivity": [0, 9.2, 3.8, 5, 5.6, 4.1, 7.1, 5.3]})

(mydata
   .query("compression > 4")                                  # keep rows
   .assign(ratio=lambda d: d.conductivity / d.compression)    # add a column
   .groupby("hydrogel_concentration", observed=True)          # define groups
   .agg(n=("ratio", "size"), mean_ratio=("ratio", "mean")))   # one row per group

## 8 · Wide to long: `.melt()`

One `.melt()` unlocks *every* gene at once: 80 × 600 = 48,000 rows, one per fish × gene.

In [ ]:
stickle_long = stickle.melt(
    id_vars=["Individual", "Genotype", "Microbiota", "Geno&Micro"],
    var_name="gene", value_name="count")
print(stickle_long.shape)
stickle_long.head(3)

In [ ]:
gene_means = (stickle_long
    .groupby(["gene", "Genotype", "Microbiota"], as_index=False)
    .agg(mean_count=("count", "mean")))
gene_means.query("gene == 'Gene1'")

## 9 · seaborn, layer 1: data + mappings + a plot kind

`sns.stripplot()` is a jittered scatter — every observation visible, none overprinting.

In [ ]:
gene1 = stickle_long.query("gene == 'Gene1'")
sns.stripplot(data=gene1, x="Genotype", y="count", jitter=0.08)
plt.show()

## 10 · Layer 2: more mappings, more layers

Mapping a column to `hue` splits the groups **and** builds the legend — automatically, from the data. Later calls draw on the same axes.

In [ ]:
sns.violinplot(data=gene1, x="Genotype", y="count", hue="Microbiota",
               inner=None, alpha=0.4, palette=["#440154", "#35b779"])
sns.stripplot(data=gene1, x="Genotype", y="count", hue="Microbiota",
              dodge=True, size=4, alpha=0.6, legend=False,
              palette="dark:black")
plt.show()

## 11 · Layers 3–5: facets, colors, labels — and `savefig()`

`col="gene"` in `sns.catplot()` gives one panel per gene for free. This is the left half of L06's destination figure.

In [ ]:
long4 = stickle_long[stickle_long["gene"].isin([f"Gene{i}" for i in range(1, 5)])]

g = sns.catplot(data=long4, x="Genotype", y="count", hue="Microbiota",
                col="gene", col_wrap=2, kind="violin", inner="point",
                palette=["#440154", "#35b779"], sharey=False, height=2.6, aspect=1.3)
g.set_axis_labels("Host genotype", "Read count")
g.set_titles("{col_name}")
g.figure.suptitle("Stickleback gut RNA-seq", y=1.02)
plt.show()

In [ ]:
Path("results").mkdir(exist_ok=True)
g.figure.savefig("results/gene_panel.svg", bbox_inches="tight")
sorted(p.name for p in Path("results").iterdir())

## 12 · The closer: all 600 genes in one figure

One mean and one variance per gene — 48,000 rows collapsed to 600 points. Dashed line: variance = mean (Poisson); real RNA-seq runs hotter.

In [ ]:
genes = [c for c in stickle.columns if c.startswith("Gene")]
mv = pd.DataFrame({"mean": stickle[genes].mean(), "var": stickle[genes].var()})

fig, ax = plt.subplots()
ax.scatter(mv["mean"], mv["var"], s=10, alpha=0.4, color="#31688e")
lims = np.array([mv["mean"].min() / 2, mv["mean"].max() * 2])
ax.plot(lims, lims, ls="--", color="black")              # variance = mean
ax.set(xscale="log", yscale="log",
       xlabel="Mean count per gene (log)", ylabel="Variance (log)",
       title="All 600 genes at once")
plt.show()

# The figure gallery

## 13 · A clustered heatmap, from synthesized expression data

`z_score=0` standardizes each gene, so the colors compare *patterns*, not levels. The dendrograms should rediscover the two modules and the control/treated split we build in.

In [ ]:
rng = np.random.default_rng(2026)
expr = pd.DataFrame(
    rng.normal(8, 0.4, size=(30, 12)) + rng.normal(0, 1, size=(30, 1)),
    index=[f"gene_{i+1}" for i in range(30)],
    columns=[f"ctrl_{i+1}" for i in range(6)] + [f"treat_{i+1}" for i in range(6)])
expr.iloc[:15, 6:] += 2.0          # module 1: up in treated
expr.iloc[15:, 6:] -= 1.5          # module 2: down in treated

sns.clustermap(expr, z_score=0, cmap="viridis", figsize=(7, 6))
plt.show()

## 14 · Growth curves with ribbons

`plt.fill_between()` draws the ribbon. The pattern works for any repeated time course: sensor logs, qPCR, cyclic loading.

In [ ]:
t = np.arange(0, 49, 2)
strains = {"wild type": (1.8, 0.25), "mutant A": (1.2, 0.35), "mutant B": (1.6, 0.15)}

fig, ax = plt.subplots()
for (name, (K, r)), c in zip(strains.items(), ["#440154", "#31688e", "#35b779"]):
    reps = np.array([K / (1 + np.exp(-r * (t - 20))) + rng.normal(0, 0.05, len(t))
                     for _ in range(4)])
    m, s = reps.mean(axis=0), reps.std(axis=0)
    ax.plot(t, m, color=c, lw=2, label=name)
    ax.fill_between(t, m - s, m + s, color=c, alpha=0.25)
ax.set(xlabel="Time (h)", ylabel="OD600",
       title="Bioreactor growth curves (mean ± SD of 4 replicates)")
ax.legend()
plt.show()

## 15 · Dose-response curves

The log-scaled x axis is what makes dose-response curves readable — doses span five orders of magnitude. The IC50 is where each curve crosses 50%.

In [ ]:
dose = np.logspace(-9, -4, 8)
fig, ax = plt.subplots()
for (name, ic50), c in zip({"compound X": 3e-7, "compound Y": 5e-6}.items(),
                           ["#440154", "#35b779"]):
    for _ in range(3):
        viab = 100 / (1 + (dose / ic50) ** 1.2) + rng.normal(0, 4, len(dose))
        ax.scatter(dose, viab, s=18, alpha=0.6, color=c)
    grid = np.logspace(-9, -4, 200)
    ax.plot(grid, 100 / (1 + (grid / ic50) ** 1.2), color=c, lw=2, label=name)
ax.set(xscale="log", xlabel="Dose (M)", ylabel="Viability (%)",
       title="Dose-response: two compounds, three replicates")
ax.legend()
plt.show()

## 16 · PCA of the real stickleback fish

PCA compresses 600 correlated gene columns into a few axes of maximal variation. `np.log1p()` tames the counts; standardizing weights every gene equally. Needs `pip install scikit-learn` (once).

In [ ]:
from sklearn.decomposition import PCA

X = np.log1p(stickle[genes].to_numpy())
X = (X - X.mean(axis=0)) / X.std(axis=0)          # like R's scale. = TRUE
pca = PCA(n_components=2).fit(X)
scores = pca.transform(X)
print("variance explained:", pca.explained_variance_ratio_.round(3))

In [ ]:
fig, ax = plt.subplots()
for grp, c in zip(sorted(stickle["Geno&Micro"].unique()),
                  ["#440154", "#31688e", "#35b779", "#fde725"]):
    sel = stickle["Geno&Micro"] == grp
    ax.scatter(scores[sel, 0], scores[sel, 1], s=30, alpha=0.8, color=c, label=grp)
ax.set(xlabel="PC1", ylabel="PC2", title="PCA of 80 fish from 600 genes")
ax.legend(title="Group")
plt.show()

## Your turn

1. Rebuild the section-11 figure for four *different* genes and `savefig()` it. A working script version is on the Examples page — but only look after your own runs.
2. Color the mean-variance plot: compute each gene's mean separately for genotypes A and B, and color each point by which genotype is higher.
3. Adapt the gallery clustermap to the real data: take the 30 most *variable* stickleback genes and cluster the 80 fish (`z_score` matters!).